In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import FactorAnalysis

# Load CSV with correct encoding
df = pd.read_csv("C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/rugby/Statistic_rugby_players.csv", encoding="latin1")

# Keep only numeric columns
num = df.select_dtypes(include=[np.number]).dropna(axis=1, how='all')

# Remove rows with missing numeric data
num_clean = num.dropna()

# ---------------------------------------------
# 1. Determine optimal number of factors
# ---------------------------------------------
corr = np.corrcoef(num_clean, rowvar=False)
eigenvalues, _ = np.linalg.eig(corr)

optimal_factors = sum(eigenvalues > 1)
print("Optimal number of factors:", optimal_factors)

# ---------------------------------------------
# 2. Fit factor analysis (using optimal factors)
# ---------------------------------------------
fa = FactorAnalysis(n_components=optimal_factors)
fa.fit(num_clean)

# Loadings matrix
loadings = pd.DataFrame(
    fa.components_.T,
    index=num_clean.columns,
    columns=[f"Factor{i+1}" for i in range(optimal_factors)]
)

print("\nFactor Loadings:")
print(loadings)

# Optional: sort loadings per factor (helpful for naming)
for i in range(optimal_factors):
    print(f"\nTop loadings for Factor {i+1}:")
    print(loadings.iloc[:, i].abs().sort_values(ascending=False).head(10))

Optimal number of factors: 9

Factor Loadings:
                     Factor1     Factor2     Factor3    Factor4    Factor5  \
racking            -7.386831   -5.381460   -6.940330  -8.615714  24.125880   
age                 0.442771   -0.173359    0.365595  -0.009580  -0.729480   
tall(m)             1.154167    4.254533    3.579692  -0.224136  -3.619389   
weight              2.019070   -1.666776   -0.024945  -7.301372  -1.191254   
start_career        0.127336    0.308295   -0.816176  -0.000837   0.369736   
club-match         -4.265314   -0.471844   -0.193582  -0.520539   0.217245   
club_W             -2.792637   -0.203834    0.193983  -0.171947  -0.052322   
club_D             -0.052666   -0.021003    0.032430  -0.025902  -0.012342   
club_L             -1.420012   -0.247007   -0.419995  -0.322691   0.281909   
club_starter       -4.217218   -0.281221   -0.370383  -0.136306  -0.010104   
club_try           -1.302330   -0.354532   -0.242840   0.384193  -0.093966   
club_points      

Factor 1 — “Overall Club Workload & Minutes Played”
Why:
Extremely high loadings on:

club_Min (338!)
other_Min
National_min
club_points, racking

This factor captures overall playing time across competitions, especially club minutes, with modest contribution from scoring.
✔ Name:
➡️  Club Workload / Playing Time Volume

Factor 2 — “International Workload & Match Exposure”
Why:
Dominated by:

National_min (252)
other_Min
club_Min (to a lesser degree)
National_Points
National_starter, National_match

This is similar to Factor 1 but national-team–focused, representing international exposure rather than club exposure.
✔ Name:
➡️  International Workload / National Team Minutes

Factor 3 — “Other‑Competition Workload”
Why:
Highest loadings on:

other_Min (178)
club_Min (lower)
other_points
other-match, other_starter

This factor isolates other competitions, distinct from club or national team.
✔ Name:
➡️  Secondary‑Competition Workload

Factor 4 — “Scoring Productivity (All Competitions)”
Why:
Dominated by scoring variables:

club_points (31.5)
National_Points
other_points
racking
also weight (high loading but less conceptually central)

This is a pure scoring output factor, representing consistent point production across all domains.
✔ Name:
➡️  Multi‑Level Scoring Productivity

Factor 5 — “General Performance Index / Match Impact”
Why:
Highly loaded on:

racking (24)
club_points, other_points
tall(m), age, etc.

This factor blends ball‑in‑play impact (racking) + scoring + anthropometrics → resembles a general performance indicator.
✔ Name:
➡️  Overall Match Impact / Performance Intensity

Factor 6 — “Anthropometrics (Size / Height Factor)”
Why:
Dominated by:

tall(m) = 18.16
smaller effects from weight, age, start_career

Height completely dominates → this is a body-size factor.
✔ Name:
➡️  Physical Size / Height Dominance

Factor 7 — “Elite Scoring Ability (National + Club)”
Why:
Strongest loadings on:

National_Points (11.6)
club_points
weight
National_try
other_points

This is similar to Factor 4 but more elite‑level scoring, especially tied to international scoring metrics.
✔ Name:
➡️  Elite Scoring Ability / Try‑Scoring Efficiency

Factor 8 — “Player Build + Club Match Activity”
Why:
Dominated by:

weight (11.5)
National_Points
club-match
club_try, club_L

This factor couples body mass with club match engagement and outcomes — often seen in forward players.
✔ Name:
➡️  Physical Power / Club Match Involvement

Factor 9 — “Opportunistic Point Contribution”
Why:
Top contributors:

other_points (9.0)
club_points
weight
racking
other_try

This factor captures sporadic scoring across competitions, reflecting opportunistic or variable contributions.
✔ Name:
➡️  Opportunistic Scoring / Secondary Point Contribution

In [2]:
import pandas as pd
import numpy as np
from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# -----------------------------------------
# 1. Load and clean numeric data
# -----------------------------------------
df = df
num = df.select_dtypes(include=[np.number]).dropna(axis=1, how='all')
num_clean = num.dropna()

# -----------------------------------------
# 2. Extract factor scores (9 factors)
# -----------------------------------------
fa = FactorAnalysis(n_components=9, random_state=0)
fa.fit(num_clean)
factor_scores = fa.transform(num_clean)

# Optional: standardize the factor scores before clustering
scaler = StandardScaler()
factor_scores_scaled = scaler.fit_transform(factor_scores)

# -----------------------------------------
# 3. Perform cluster analysis
# -----------------------------------------
kmeans = KMeans(n_clusters=4, random_state=0)
clusters = kmeans.fit_predict(factor_scores_scaled)

# Attach cluster labels to the dataframe
clustered_df = num_clean.copy()
clustered_df["cluster"] = clusters

print(clustered_df.head())

# -----------------------------------------
# 4. (Optional) Print cluster sizes
# -----------------------------------------
print("Cluster sizes:")
print(pd.Series(clusters).value_counts())

   racking  age  tall(m)  weight  start_career  club-match  club_W  club_D  \
0        1   27     1.74      86        2000.0        14.0    11.0     0.0   
1        2   30     1.90     103        2013.0         0.0     0.0     0.0   
3        4   26   193.00     106        2017.0         7.0     5.0     0.0   
4        5   32     2.03     126        2013.0         7.0     4.0     0.0   
5        6   25     1.91     110        2021.0        10.0     8.0     0.0   

   club_L  club_starter  ...  National_W  National_D  National_L  \
0     3.0           9.0  ...           2           0           0   
1     0.0           0.0  ...           9           0           4   
3     2.0           6.0  ...          12           0           2   
4     3.0           7.0  ...          10           0           2   
5     2.0           5.0  ...           7           0           3   

   National_starter  National_try  National_Points  National_min  yellow card  \
0                 2             0        

C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


⭐ Cluster Names & Descriptions
Cluster 0 — “High‑Workload Club Specialists”
This cluster represents players who accumulate substantial minutes in club competitions and sustain consistently high on‑field involvement.
Their main strength is high playing time volume, indicating reliability, endurance, and continuous contribution in club settings.
A key weakness is comparatively lower international exposure, suggesting they may not be primary selections for national‑team roles.

Cluster 1 — “International Impact Leaders”
This cluster contains players whose profiles show strong national‑team minutes, scoring contributions, and international match engagement.
They excel in international productivity, especially through national minutes, points, and starter status.
Their weakness is that they generally show lower involvement in club or secondary‑competition workloads, implying a narrower competition footprint.

Cluster 2 — “Balanced All‑Round Contributors”
These players exhibit a balanced profile across club, national, and other‑competition factors without extreme highs or lows.
Their strength lies in versatility, maintaining moderate workload, scoring contributions, and physical attributes across multiple contexts.
Their weakness is the absence of standout dominance in any single factor, which may reduce differentiation for selection in specialized roles.

Cluster 3 — “Physical Power & Opportunistic Scorers”
This group is characterized by strong loadings on body mass–related factors and secondary scoring contributions.
They excel in strength‑related metrics and opportunistic scoring across non‑primary competitions, which can make them valuable impact players.
Their weakness is lower sustained workload and reduced national‑team involvement, limiting their overall match influence.

In [3]:
import pandas as pd
import numpy as np

# Load data
df = df

# Helper to compute descending ranks (1 = best/highest)
def rank_desc(series: pd.Series) -> pd.Series:
    return series.rank(ascending=False, method="min")

# Select Antoine Dupont (case-insensitive, trimmed)
player_mask = df["Name"].str.strip().str.lower() == "antoine dupont"
player = df.loc[player_mask].iloc[0]

# Metrics to compare and label
metrics = {
    "club_Min": "Club minutes",
    "club_points": "Club points",
    "club_try": "Club tries",
    "club-match": "Club matches",
    "National_min": "National minutes",
    "National_Points": "National points",
    "National_try": "National tries",
    "other_Min": "Other-competition minutes",
    "other_points": "Other-competition points",
}

# Compute ranks across the whole dataset (higher=better for these positive metrics)
ranks = {}
for col in metrics:
    if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
        ranks[col] = int(rank_desc(df[col]).loc[player.name])

n_players = len(df)

# Print a compact summary
print(f"Player: {player['Name']} (n={n_players} players)")
for col, label in metrics.items():
    if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
        val = player[col]
        print(f"{label}: {val}  |  Rank: #{ranks[col]} of {n_players}")

# Example: derive key talking points
strengths = {
    "Other-competition minutes": (player.get("other_Min", np.nan), ranks.get("other_Min")),
    "Club tries": (player.get("club_try", np.nan), ranks.get("club_try")),
    "Club points": (player.get("club_points", np.nan), ranks.get("club_points")),
}

weaknesses = {
    "National minutes": (player.get("National_min", np.nan), ranks.get("National_min")),
    "National points": (player.get("National_Points", np.nan), ranks.get("National_Points")),
    "National tries": (player.get("National_try", np.nan), ranks.get("National_try")),
}

print("\nTop strengths (value, rank):", strengths)
print("Top weaknesses (value, rank):", weaknesses)


Player: Antoine Dupont (n=99 players)
Club minutes: 866.0  |  Rank: #34 of 99
Club points: 30.0  |  Rank: #22 of 99
Club tries: 6.0  |  Rank: #9 of 99
Club matches: 14.0  |  Rank: #19 of 99
National minutes: 127  |  Rank: #97 of 99
National points: 0  |  Rank: #80 of 99
National tries: 0  |  Rank: #77 of 99
Other-competition minutes: 627.0  |  Rank: #2 of 99
Other-competition points: 25.0  |  Rank: #11 of 99

Top strengths (value, rank): {'Other-competition minutes': (np.float64(627.0), 2), 'Club tries': (np.float64(6.0), 9), 'Club points': (np.float64(30.0), 22)}
Top weaknesses (value, rank): {'National minutes': (np.int64(127), 97), 'National points': (np.int64(0), 80), 'National tries': (np.int64(0), 77)}


# Summary from the chat
Strengths: Dupont shows exceptional workload outside of domestic league play (ranked #2 for other‑competition minutes at 627), strong finishing at club level (club tries 6, ranked #9), and solid club contribution (club points 30, ranked #22 out of 99).
Average/Weaknesses: His international usage and output are comparatively low (National_min 127, ranked #97; National_points 0, ranked #80; National_try 0, ranked #77), indicating limited recent national‑team impact in this sample.
Conclusion: Overall, he profiles as a high‑involvement, high‑impact club and European‑competition scrum‑half with strong try‑scoring for his club but relatively modest current international contribution.

## Code after point out copilot did not use FA and clustering

In [4]:
import pandas as pd
import numpy as np
from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------------
# 1. Load & prepare numeric data
# --------------------------------------------------------
df = df
num = df.select_dtypes(include=[np.number]).dropna(axis=1, how="all")
num_clean = num.dropna()

# --------------------------------------------------------
# 2. Factor Analysis (9 factors)
# --------------------------------------------------------
fa = FactorAnalysis(n_components=9, random_state=0)
fa.fit(num_clean)
factors = fa.transform(num_clean)

# Standardize factor scores before clustering
scaler = StandardScaler()
factors_scaled = scaler.fit_transform(factors)

# --------------------------------------------------------
# 3. Re-run clustering (k=4)
# --------------------------------------------------------
kmeans = KMeans(n_clusters=4, random_state=0)
cluster_labels = kmeans.fit_predict(factors_scaled)

# Attach cluster labels to players
clustered = df.loc[num_clean.index].copy()
clustered["cluster"] = cluster_labels

# --------------------------------------------------------
# 4. Identify Antoine Dupont
# --------------------------------------------------------
dupont_row = clustered[clustered["Name"].str.lower().str.strip() == "antoine dupont"]
dupont_cluster = int(dupont_row["cluster"].iloc[0])

# --------------------------------------------------------
# 5. Cluster descriptions (same as earlier analysis)
# --------------------------------------------------------
cluster_descriptions = {
    0: {
        "strength": "high club workload, strong playing time volume, and consistent involvement across club competitions.",
        "weakness": "comparatively lower involvement and scoring impact in international matches.",
        "summary": "a high-involvement club performer with strong consistency but modest international influence."
    },
    1: {
        "strength": "strong international workload, national minutes, and scoring contributions.",
        "weakness": "lower club or secondary-competition involvement relative to international specialists.",
        "summary": "a player whose primary impact is at the international level."
    },
    2: {
        "strength": "balanced contribution across club, national, and other competitions.",
        "weakness": "lacks standout dominance in any single domain.",
        "summary": "a well-rounded but not highly specialized contributor."
    },
    3: {
        "strength": "strong physical metrics and opportunistic scoring ability, especially outside primary competitions.",
        "weakness": "lower sustained workload and reduced national-team involvement.",
        "summary": "a physically strong player with opportunistic scoring rather than consistent match influence."
    }
}

# --------------------------------------------------------
# 6. Build the three-sentence summary for Dupont
# --------------------------------------------------------
c = cluster_descriptions[dupont_cluster]

summary_sentences = [
    f"Antoine Dupont’s strengths align with his cluster profile, showing {c['strength']}",
    f"He is average or weaker in areas characterized by {c['weakness']}",
    f"Overall, Antoine Dupont is {c['summary']}"
]

for s in summary_sentences:
    print(s)

Antoine Dupont’s strengths align with his cluster profile, showing high club workload, strong playing time volume, and consistent involvement across club competitions.
He is average or weaker in areas characterized by comparatively lower involvement and scoring impact in international matches.
Overall, Antoine Dupont is a high-involvement club performer with strong consistency but modest international influence.


C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
